In [1]:
import sys

!{sys.executable} -m pip install -U xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 2.3 MB/s  0:00:012.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.0 MB/s  0:00:00m 3.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [lightgbm]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip


In [5]:
feature_sets
y_train
y_validation
y_test
y_train_log
evaluate_predictions
MODEL_ROOT
REPORT_ROOT
test_df

,asin,title,category_name,price,listPrice,stars,reviews,isBestSeller,boughtInLastMonth,product_text,...,height,file_size_bytes,image_exists,model_text,image_file_exists,original_split_group,split_group,dataset_split,reviews_log1p,bought_log1p
0,B01N52WJRT,Channellock - 6 Xtra Slim Jaw Adj Wrench (806SW),Industrial Power & Hand Tools,21.99,29.08,4.6,0,0,100,title: Channellock - 6 Xtra Slim Jaw Adj Wrenc...,...,320.0,4180.0,True,title: Channellock - 6 Xtra Slim Jaw Adj Wrenc...,True,cluster_85__price_medium,cluster_85__price_medium,test,0.000000,4.615121
1,B0BLTDC4HF,Boys' Short Sleeve Swim Rashguard with UPF 50+...,Boys' Clothing,8.39,0.00,5.0,2,0,0,title: Boys' Short Sleeve Swim Rashguard with ...,...,320.0,8981.0,True,title: Boys' Short Sleeve Swim Rashguard with ...,True,cluster_98__price_very_low,cluster_98__price_very_low,test,1.098612,0.000000
2,B07RBL5C1Q,8mm Digital Precise Portable Color Analyzer Co...,Lab & Scientific Products,238.00,0.00,4.3,0,0,0,title: 8mm Digital Precise Portable Color Anal...,...,320.0,8823.0,True,title: 8mm Digital Precise Portable Color Anal...,True,cluster_9__price_luxury,cluster_9__price_luxury,test,0.000000,0.000000
3,B0CBLRT782,"doepeBAE Cat Ear Headphones with Microphone,Bl...",Headphones & Earbuds,15.99,0.00,0.0,0,0,0,title: doepeBAE Cat Ear Headphones with Microp...,...,320.0,8453.0,True,title: doepeBAE Cat Ear Headphones with Microp...,True,cluster_66__price_low,cluster_66__price_low,test,0.000000,0.000000
4,B0BLB2XF6W,"Uniheat Shipping Warmers, The Countdown: Four ...",Reptiles & Amphibian Supplies,23.97,0.00,5.0,0,0,0,"title: Uniheat Shipping Warmers, The Countdown...",...,213.0,17117.0,True,"title: Uniheat Shipping Warmers, The Countdown...",True,cluster_29__price_medium,cluster_29__price_medium,test,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2992,B00V0NK96Q,Men's 3-Pack Cotton Square Neck Tank Top,Men's Clothing,16.76,32.00,4.1,0,0,100,title: Men's 3-Pack Cotton Square Neck Tank To...,...,320.0,5600.0,True,title: Men's 3-Pack Cotton Square Neck Tank To...,True,cluster_56__price_medium,cluster_56__price_medium,test,0.000000,4.615121
2993,B08ZXQ6BT5,Black and White Canvas Art Wall Decor for Livi...,Wall Art,69.99,0.00,4.0,0,0,0,title: Black and White Canvas Art Wall Decor f...,...,150.0,14431.0,True,title: Black and White Canvas Art Wall Decor f...,True,cluster_6__price_premium,cluster_6__price_premium,test,0.000000,0.000000
2994,B0BZRZXDRJ,Women's Mid Waist Flare Pants Straight Leg Dre...,Women's Clothing,44.99,0.00,5.0,0,0,0,title: Women's Mid Waist Flare Pants Straight ...,...,320.0,8049.0,True,title: Women's Mid Waist Flare Pants Straight ...,True,cluster_67__price_premium,cluster_67__price_premium,test,0.000000,0.000000
2995,B0C23BN1H7,JIAHANG Baby Girl Halloween Hair Bow Clips wit...,Baby Care Products,6.99,0.00,0.0,0,0,0,title: JIAHANG Baby Girl Halloween Hair Bow Cl...,...,320.0,22908.0,True,title: JIAHANG Baby Girl Halloween Hair Bow Cl...,True,cluster_66__price_very_low,cluster_66__price_very_low,test,0.000000,0.000000


In [6]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# PATHS
# ============================================================

CURRENT_DIRECTORY = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name.lower() == "notebooks"
    else CURRENT_DIRECTORY
)

MODEL_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "model_input"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT
    / "data"
    / "amazon_multimodal"
    / "embeddings"
    / "clip"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "price_prediction"
)

REPORT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "price_prediction"
)

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD SPLIT
# ============================================================

def load_split(split_name):

    image_embeddings = np.load(
        EMBEDDING_ROOT
        / f"{split_name}_image_embeddings.npy"
    )

    text_embeddings = np.load(
        EMBEDDING_ROOT
        / f"{split_name}_text_embeddings.npy"
    )

    metadata = pd.read_parquet(
        MODEL_INPUT_ROOT
        / f"{split_name}.parquet"
    ).reset_index(drop=True)

    if not (
        len(image_embeddings)
        == len(text_embeddings)
        == len(metadata)
    ):
        raise ValueError(
            f"{split_name} row mismatch: "
            f"image={len(image_embeddings)}, "
            f"text={len(text_embeddings)}, "
            f"metadata={len(metadata)}"
        )

    return (
        image_embeddings.astype(np.float32),
        text_embeddings.astype(np.float32),
        metadata,
    )


# ============================================================
# LOAD TRAIN / VALIDATION / TEST
# ============================================================

X_train_image, X_train_text, train_df = load_split(
    "train"
)

X_val_image, X_val_text, validation_df = load_split(
    "validation"
)

X_test_image, X_test_text, test_df = load_split(
    "test"
)

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))


# ============================================================
# TARGET
# ============================================================

y_train = (
    pd.to_numeric(
        train_df["price"],
        errors="raise",
    )
    .astype(np.float32)
    .to_numpy()
)

y_validation = (
    pd.to_numeric(
        validation_df["price"],
        errors="raise",
    )
    .astype(np.float32)
    .to_numpy()
)

y_test = (
    pd.to_numeric(
        test_df["price"],
        errors="raise",
    )
    .astype(np.float32)
    .to_numpy()
)

y_train_log = np.log1p(y_train)


# ============================================================
# STRUCTURED FEATURE ENGINEERING
# ============================================================

for dataframe in [
    train_df,
    validation_df,
    test_df,
]:

    dataframe["reviews_log1p"] = np.log1p(
        pd.to_numeric(
            dataframe["reviews"],
            errors="coerce",
        ).fillna(0)
    )

    dataframe["bought_log1p"] = np.log1p(
        pd.to_numeric(
            dataframe["boughtInLastMonth"],
            errors="coerce",
        ).fillna(0)
    )


numeric_columns = [
    "stars",
    "reviews_log1p",
    "bought_log1p",
    "isBestSeller",
    "cluster_id",
]

categorical_columns = [
    "category_name",
    "price_band",
]


# ============================================================
# PREPROCESSOR
# ============================================================

PREPROCESSOR_FILE = (
    MODEL_ROOT
    / "structured_preprocessor.joblib"
)

if PREPROCESSOR_FILE.exists():

    print("Loading existing structured preprocessor...")

    structured_preprocessor = joblib.load(
        PREPROCESSOR_FILE
    )

    X_train_structured = (
        structured_preprocessor
        .transform(train_df)
        .astype(np.float32)
    )

else:

    print("Creating structured preprocessor...")

    structured_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric_columns,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_columns,
            ),
        ],
        remainder="drop",
    )

    X_train_structured = (
        structured_preprocessor
        .fit_transform(train_df)
        .astype(np.float32)
    )

    joblib.dump(
        structured_preprocessor,
        PREPROCESSOR_FILE,
    )


X_validation_structured = (
    structured_preprocessor
    .transform(validation_df)
    .astype(np.float32)
)

X_test_structured = (
    structured_preprocessor
    .transform(test_df)
    .astype(np.float32)
)


# ============================================================
# RECREATE FEATURE SETS
# ============================================================

feature_sets = {

    "structured_only": (
        X_train_structured,
        X_validation_structured,
        X_test_structured,
    ),

    "text_only": (
        X_train_text,
        X_val_text,
        X_test_text,
    ),

    "image_only": (
        X_train_image,
        X_val_image,
        X_test_image,
    ),

    "multimodal_fusion": (

        np.concatenate(
            [
                X_train_image,
                X_train_text,
                X_train_structured,
            ],
            axis=1,
        ),

        np.concatenate(
            [
                X_val_image,
                X_val_text,
                X_validation_structured,
            ],
            axis=1,
        ),

        np.concatenate(
            [
                X_test_image,
                X_test_text,
                X_test_structured,
            ],
            axis=1,
        ),
    ),
}


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_predictions(
    model_name,
    actual_price,
    predicted_price,
):

    return {

        "model": model_name,

        "mae": mean_absolute_error(
            actual_price,
            predicted_price,
        ),

        "rmse": np.sqrt(
            mean_squared_error(
                actual_price,
                predicted_price,
            )
        ),

        "median_absolute_error":
            median_absolute_error(
                actual_price,
                predicted_price,
            ),

        "r2": r2_score(
            actual_price,
            predicted_price,
        ),
    }


# ============================================================
# VERIFY EVERYTHING
# ============================================================

print()
print("=" * 80)
print("FEATURE SETS RECREATED")
print("=" * 80)

for name, (
    train_features,
    validation_features,
    test_features,
) in feature_sets.items():

    print(
        f"{name:20s}",
        "train:",
        train_features.shape,
        "validation:",
        validation_features.shape,
        "test:",
        test_features.shape,
    )

print()
print("Target shapes:")
print("Train:", y_train.shape)
print("Validation:", y_validation.shape)
print("Test:", y_test.shape)

print()
print("✅ Ready for XGBoost + LightGBM")

Train: 13984
Validation: 2997
Test: 2997
Loading existing structured preprocessor...

FEATURE SETS RECREATED
structured_only      train: (13984, 251) validation: (2997, 251) test: (2997, 251)
text_only            train: (13984, 512) validation: (2997, 512) test: (2997, 512)
image_only           train: (13984, 512) validation: (2997, 512) test: (2997, 512)
multimodal_fusion    train: (13984, 1275) validation: (2997, 1275) test: (2997, 1275)

Target shapes:
Train: (13984,)
Validation: (2997,)
Test: (2997,)

✅ Ready for XGBoost + LightGBM


In [10]:
from __future__ import annotations

import gc
import json

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb


# ============================================================
# OUTPUT FILES
# ============================================================

BOOSTING_RESULTS_FILE = (
    REPORT_ROOT
    / "boosting_model_comparison.csv"
)

BOOSTING_PREDICTIONS_FILE = (
    REPORT_ROOT
    / "boosting_test_predictions.csv"
)

BEST_BOOSTING_MODEL_FILE = (
    MODEL_ROOT
    / "best_boosting_model.joblib"
)

BEST_BOOSTING_CONFIG_FILE = (
    MODEL_ROOT
    / "best_boosting_model_config.json"
)


# ============================================================
# VALIDATE VARIABLES
# ============================================================

required_variables = [
    "feature_sets",
    "y_train",
    "y_validation",
    "y_test",
    "y_train_log",
    "evaluate_predictions",
    "MODEL_ROOT",
    "REPORT_ROOT",
    "test_df",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run the feature-preparation / Random Forest cell first.\n"
        f"Missing variables: {missing_variables}"
    )


# ============================================================
# TARGETS
# ============================================================

y_validation_log = np.log1p(
    y_validation
)

y_test_log = np.log1p(
    y_test
)


# ============================================================
# HELPERS
# ============================================================

def restore_price(
    log_predictions: np.ndarray,
) -> np.ndarray:

    predictions = np.expm1(
        np.asarray(
            log_predictions,
            dtype=np.float64,
        )
    )

    return np.clip(
        predictions,
        0,
        None,
    )


def print_result(
    title: str,
    metrics: dict,
):

    print()
    print(title)
    print("-" * 60)

    print(
        "MAE:",
        round(metrics["mae"], 4),
    )

    print(
        "RMSE:",
        round(metrics["rmse"], 4),
    )

    print(
        "Median AE:",
        round(
            metrics[
                "median_absolute_error"
            ],
            4,
        ),
    )

    print(
        "R²:",
        round(metrics["r2"], 4),
    )


# ============================================================
# RESULT CONTAINERS
# ============================================================

results = []

trained_models = {}

test_prediction_map = {}


# ============================================================
# TRAIN ALL FEATURE SETS
# ============================================================

for feature_name, (
    X_train,
    X_validation,
    X_test,
) in feature_sets.items():

    print()
    print("=" * 80)
    print(
        f"FEATURE SET: {feature_name}"
    )
    print("=" * 80)

    X_train = np.asarray(
        X_train,
        dtype=np.float32,
    )

    X_validation = np.asarray(
        X_validation,
        dtype=np.float32,
    )

    X_test = np.asarray(
        X_test,
        dtype=np.float32,
    )


    # ========================================================
    # XGBOOST
    # ========================================================

    print("\nTraining XGBoost...")

    xgb_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",

        n_estimators=2500,
        learning_rate=0.03,

        max_depth=8,
        min_child_weight=4,

        subsample=0.85,
        colsample_bytree=0.80,

        reg_alpha=0.05,
        reg_lambda=1.0,

        tree_method="hist",

        early_stopping_rounds=100,

        random_state=42,
        n_jobs=-1,
    )

    xgb_model.fit(
        X_train,
        y_train_log,

        eval_set=[
            (
                X_validation,
                y_validation_log,
            )
        ],

        verbose=False,
    )

    xgb_validation_prediction = (
        restore_price(
            xgb_model.predict(
                X_validation
            )
        )
    )

    xgb_test_prediction = (
        restore_price(
            xgb_model.predict(
                X_test
            )
        )
    )

    xgb_validation_metrics = (
        evaluate_predictions(
            f"xgboost_{feature_name}_validation",
            y_validation,
            xgb_validation_prediction,
        )
    )

    xgb_test_metrics = (
        evaluate_predictions(
            f"xgboost_{feature_name}_test",
            y_test,
            xgb_test_prediction,
        )
    )

    results.extend(
        [
            xgb_validation_metrics,
            xgb_test_metrics,
        ]
    )

    xgb_name = (
        f"xgboost_{feature_name}"
    )

    trained_models[
        xgb_name
    ] = xgb_model

    test_prediction_map[
        xgb_name
    ] = xgb_test_prediction

    print_result(
        "XGBoost Validation",
        xgb_validation_metrics,
    )

    if hasattr(
        xgb_model,
        "best_iteration",
    ):
        print(
            "Best iteration:",
            xgb_model.best_iteration,
        )


    # ========================================================
    # LIGHTGBM
    # ========================================================

    print("\nTraining LightGBM...")

    lightgbm_model = (
        lgb.LGBMRegressor(
            objective="regression",

            n_estimators=3000,
            learning_rate=0.025,

            num_leaves=63,
            max_depth=-1,

            min_child_samples=25,

            subsample=0.85,
            colsample_bytree=0.80,

            reg_alpha=0.05,
            reg_lambda=1.0,

            random_state=42,
            n_jobs=-1,

            verbosity=-1,
        )
    )

    lightgbm_model.fit(
        X_train,
        y_train_log,

        eval_set=[
            (
                X_validation,
                y_validation_log,
            )
        ],

        eval_metric="rmse",

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False,
            ),

            lgb.log_evaluation(
                period=0
            ),
        ],
    )

    lightgbm_validation_prediction = (
        restore_price(
            lightgbm_model.predict(
                X_validation,
                num_iteration=(
                    lightgbm_model.best_iteration_
                ),
            )
        )
    )

    lightgbm_test_prediction = (
        restore_price(
            lightgbm_model.predict(
                X_test,
                num_iteration=(
                    lightgbm_model.best_iteration_
                ),
            )
        )
    )

    lightgbm_validation_metrics = (
        evaluate_predictions(
            f"lightgbm_{feature_name}_validation",
            y_validation,
            lightgbm_validation_prediction,
        )
    )

    lightgbm_test_metrics = (
        evaluate_predictions(
            f"lightgbm_{feature_name}_test",
            y_test,
            lightgbm_test_prediction,
        )
    )

    results.extend(
        [
            lightgbm_validation_metrics,
            lightgbm_test_metrics,
        ]
    )

    lightgbm_name = (
        f"lightgbm_{feature_name}"
    )

    trained_models[
        lightgbm_name
    ] = lightgbm_model

    test_prediction_map[
        lightgbm_name
    ] = lightgbm_test_prediction

    print_result(
        "LightGBM Validation",
        lightgbm_validation_metrics,
    )

    print(
        "Best iteration:",
        lightgbm_model.best_iteration_,
    )

    gc.collect()


# ============================================================
# CREATE RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

results_df.to_csv(
    BOOSTING_RESULTS_FILE,
    index=False,
)


validation_results_df = (
    results_df[
        results_df["model"]
        .str.endswith(
            "_validation"
        )
    ]
    .sort_values(
        by="mae",
        ascending=True,
    )
    .reset_index(drop=True)
)


print()
print("=" * 80)
print("VALIDATION MODEL RANKING")
print("=" * 80)

display(
    validation_results_df
)


# ============================================================
# SELECT BEST MODEL
# ============================================================

best_row = (
    validation_results_df
    .iloc[0]
)

best_model_name = (
    str(
        best_row["model"]
    )
    .replace(
        "_validation",
        "",
    )
)

best_model = (
    trained_models[
        best_model_name
    ]
)


joblib.dump(
    best_model,
    BEST_BOOSTING_MODEL_FILE,
)


print()
print("=" * 80)
print("BEST BOOSTING MODEL")
print("=" * 80)

print(
    "Model:",
    best_model_name,
)

print(
    "Validation MAE:",
    round(
        best_row["mae"],
        4,
    ),
)

print(
    "Validation RMSE:",
    round(
        best_row["rmse"],
        4,
    ),
)

print(
    "Validation Median AE:",
    round(
        best_row[
            "median_absolute_error"
        ],
        4,
    ),
)

print(
    "Validation R²:",
    round(
        best_row["r2"],
        4,
    ),
)


# ============================================================
# TEST PREDICTION REPORT
# ============================================================

prediction_columns = [
    "asin",
    "title",
    "category_name",
    "price",
    "price_band",
    "cluster_id",
]

prediction_columns = [
    column
    for column in prediction_columns
    if column in test_df.columns
]


prediction_df = (
    test_df[
        prediction_columns
    ]
    .copy()
)


for model_name, predictions in (
    test_prediction_map.items()
):

    prediction_df[
        f"{model_name}_predicted_price"
    ] = predictions

    prediction_df[
        f"{model_name}_absolute_error"
    ] = np.abs(
        y_test - predictions
    )


prediction_df.to_csv(
    BOOSTING_PREDICTIONS_FILE,
    index=False,
)


# ============================================================
# SAVE CONFIG
# ============================================================

config = {
    "target": "log1p(price)",

    "algorithms": [
        "XGBoost",
        "LightGBM",
    ],

    "feature_sets": [
        "structured_only",
        "text_only",
        "image_only",
        "multimodal_fusion",
    ],

    "selection_metric": (
        "validation_mae"
    ),

    "best_model": (
        best_model_name
    ),

    "best_validation_mae": float(
        best_row["mae"]
    ),

    "best_validation_rmse": float(
        best_row["rmse"]
    ),

    "best_validation_r2": float(
        best_row["r2"]
    ),
}


with BEST_BOOSTING_CONFIG_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        config,
        file,
        indent=2,
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 80)
print(
    "XGBOOST + LIGHTGBM TRAINING COMPLETED"
)
print("=" * 80)

print(
    "Results saved:",
    BOOSTING_RESULTS_FILE,
)

print(
    "Predictions saved:",
    BOOSTING_PREDICTIONS_FILE,
)

print(
    "Best model saved:",
    BEST_BOOSTING_MODEL_FILE,
)


FEATURE SET: structured_only

Training XGBoost...

XGBoost Validation
------------------------------------------------------------
MAE: 11.0995
RMSE: 77.1668
Median AE: 2.1864
R²: 0.5213
Best iteration: 687

Training LightGBM...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



LightGBM Validation
------------------------------------------------------------
MAE: 10.9925
RMSE: 76.1688
Median AE: 2.1275
R²: 0.5336
Best iteration: 244

FEATURE SET: text_only

Training XGBoost...

XGBoost Validation
------------------------------------------------------------
MAE: 26.2291
RMSE: 99.3909
Median AE: 8.5528
R²: 0.2058
Best iteration: 2290

Training LightGBM...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



LightGBM Validation
------------------------------------------------------------
MAE: 25.8866
RMSE: 98.4187
Median AE: 8.4365
R²: 0.2212
Best iteration: 2196

FEATURE SET: image_only

Training XGBoost...

XGBoost Validation
------------------------------------------------------------
MAE: 28.7624
RMSE: 103.3682
Median AE: 9.8941
R²: 0.141
Best iteration: 1696

Training LightGBM...


...


LightGBM Validation
------------------------------------------------------------
MAE: 28.4758
RMSE: 103.1632
Median AE: 9.9225
R²: 0.1444
Best iteration: 1891

FEATURE SET: multimodal_fusion

Training XGBoost...

XGBoost Validation
------------------------------------------------------------
MAE: 11.9191
RMSE: 79.7898
Median AE: 2.3267
R²: 0.4882
Best iteration: 460

Training LightGBM...


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)



LightGBM Validation
------------------------------------------------------------
MAE: 11.7431
RMSE: 79.5119
Median AE: 2.2721
R²: 0.4917
Best iteration: 504

VALIDATION MODEL RANKING


,model,mae,rmse,median_absolute_error,r2
0,lightgbm_structured_only_validation,10.992532,76.168752,2.127532,0.533558
1,xgboost_structured_only_validation,11.099468,77.166761,2.186389,0.521255
2,lightgbm_multimodal_fusion_validation,11.743144,79.511946,2.272110,0.491714
3,xgboost_multimodal_fusion_validation,11.919139,79.789847,2.326717,0.488154
4,lightgbm_text_only_validation,25.886587,98.418694,8.436498,0.221249
5,xgboost_text_only_validation,26.229126,99.390925,8.552797,0.205787
6,lightgbm_image_only_validation,28.475819,103.163169,9.922479,0.144356
7,xgboost_image_only_validation,28.762368,103.368250,9.894130,0.140951



BEST BOOSTING MODEL
Model: lightgbm_structured_only
Validation MAE: 10.9925
Validation RMSE: 76.1688
Validation Median AE: 2.1275
Validation R²: 0.5336

XGBOOST + LIGHTGBM TRAINING COMPLETED
Results saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/boosting_model_comparison.csv
Predictions saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/data/reports/price_prediction/boosting_test_predictions.csv
Best model saved: /Users/souravkumar/Downloads/AI-Powered-Smart-Product-Pricing-Visual-Attribute-Extraction-System/models/price_prediction/best_boosting_model.joblib
